# VDB
(lucas)

In [ ]:
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import zipfile
import tempfile
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

class vdb:
    def __init__(self, model_name='intfloat/e5-large-v2', device=None, verbose=False):
        """
        Initialize the vector database with the specified embedding model.
        E-01: Build embed_texts() — e5-large-v2 setup on GPU.

        Args:
            model_name (str): The name of the sentence-transformers model to use.
            device (str, optional): The device to run the model on ('cuda', 'mps', 'cpu'). Auto-detected if None.
            verbose (bool): Whether to print verbose output statements during operations.

        Returns:
            None
        """
        if device is None:
            # Automatically detect GPU if available
            self.device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
        else:
            self.device = device

        self.verbose = verbose
        self.model = SentenceTransformer(model_name, device=self.device)
        self.dim = 1024  # e5-large-v2 embedding dimension
        self.index = None
        self.chunks = []

        self.vprint(f"[VDB.__init__] Model {model_name} loaded on {self.device}.")

    def _embed_texts(self, texts, prefix, batch_size=32, normalize_embeddings=True):
        """
        E-01 & E-02 & E-03: Embed texts with appropriate prefix.
        Batch embed all chunks (batch_size=32, normalize_embeddings=True).

        Args:
            texts (list of str): The list of text strings to embed.
            prefix (str): The prefix to prepend to each text before embedding (e.g., 'passage: ', 'query: ').
            batch_size (int): The batch size for encoding.
            normalize_embeddings (bool): Whether to normalize the resulting embeddings.

        Returns:
            numpy.ndarray: The generated embeddings.
        """
        self.vprint(f"[VDB._embed_texts] Embedding {len(texts)} texts with prefix '{prefix}'...")

        prefixed_texts = [f"{prefix}{text}" for text in texts]
        embeddings = self.model.encode(
            prefixed_texts,
            batch_size=batch_size,
            normalize_embeddings=normalize_embeddings,
            convert_to_numpy=True,
            show_progress_bar=True
        )
        return embeddings

    def _build_index(self, embeddings, index_type='ivfflat', nlist=100):
        """
        E-04: Build FAISS IndexFlatIP from embeddings (dim=1024)
        E-08: Switch to IVFFlat index for HuggingFace Spaces 16 GB RAM cap

        Args:
            embeddings (numpy.ndarray): The embeddings to add to the index.
            index_type (str): The type of FAISS index to build ('flat' or 'ivfflat').
            nlist (int): The number of cells/clusters for IVFFlat index.
        """

        self.vprint(f"[VDB._build_index] Building index of type '{index_type}' with {len(embeddings)} embeddings...")

        if self.index is not None:
            print("[VDB._build_index] WARNING: Index already exists. Index will be overwritten.")

        if index_type.lower() == 'flat':
            # E-04: FAISS IndexFlatIP
            self.index = faiss.IndexFlatIP(self.dim)
            self.index.add(embeddings)

        elif index_type.lower() == 'ivfflat':
            # E-08: FAISS IndexIVFFlat
            quantizer = faiss.IndexFlatIP(self.dim)
            self.index = faiss.IndexIVFFlat(quantizer, self.dim, nlist, faiss.METRIC_INNER_PRODUCT)

            # IVFFlat index must be trained before adding data
            if not self.index.is_trained:
                self.vprint("[VDB.build_index] Training IVFFlat index...")

                try:
                    self.index.train(embeddings)
                except Exception as e:
                    print("[VDB.build_index] ERROR: Failed to train IVFFlat index. reset to None:", e)
                    self.index = None


            self.index.add(embeddings)
        else:
            raise ValueError(f"Unsupported index type: {index_type}")

    def save_index(self, index_path, chunks_path):
        """
        E-05: Save index to Drive with faiss.write_index()

        Args:
            index_path (str): The file path where the FAISS index will be saved.
            chunks_path (str): The file path where the chunks JSON will be saved.
        """
        if self.index is None:
            raise ValueError("No index to save. Please build the index first.")

        self.vprint(f"[VDB.save_index] Saving index to {index_path}...")
        faiss.write_index(self.index, index_path)

        self.vprint(f"[VDB.save_index] Saving chunks to {chunks_path}...")
        with open(chunks_path, 'w', encoding='utf-8') as f:
            json.dump(self.chunks, f, ensure_ascii=False, indent=2)

    def load_index(self, index_path, chunks_path):
        """
        E-06: Build index loader — faiss.read_index() + load chunks.json

        Args:
            index_path (str): The file path from which the FAISS index will be loaded.
            chunks_path (str): The file path from which the chunks JSON will be loaded.
        """
        self.vprint(f"[VDB.load_index] Loading index from {index_path}...")
        self.index = faiss.read_index(index_path)

        self.vprint(f"[VDB.load_index] Loading chunks from {chunks_path}...")
        with open(chunks_path, 'r', encoding='utf-8') as f:
            self.chunks = json.load(f)

    def add_documents(self, chunks, index_type='ivfflat', nlist=100):
        """
        Process a list of chunks, embed them, and add to the index.
        `chunks` should be a list of dicts, e.g., [{"text": "...", "metadata": {...}}, ...]

        Args:
            chunks (list of dict or str): The documents to add to the database.
            index_type (str): The type of FAISS index to build if one doesn't exist ('flat' or 'ivfflat').
            nlist (int): The number of cells/clusters for IVFFlat index.
        """

        self.vprint(f"[VDB.add_documents] Adding {len(chunks)} documents...")

        self.chunks.extend(chunks)
        texts = [chunk['text'] if isinstance(chunk, dict) else chunk for chunk in chunks]

        # E-02: Add e5 prefixes correctly (passage: for chunks)
        self.vprint(f"[VDB.add_documents] Embedding {len(texts)} passages...")
        embeddings = self._embed_texts(texts, prefix="passage: ", batch_size=32, normalize_embeddings=True)

        if self.index is None:
            self.vprint(f"[VDB.add_documents] Building {index_type} index...")
            # If dataset is smaller than nlist, faiss will throw error for IVFFlat, so we handle it
            if index_type == 'ivfflat' and len(embeddings) < nlist:
                self.vprint(f"[VDB.add_documents] Warning: Not enough embeddings ({len(embeddings)}) for nlist={nlist}. Falling back to Flat index.")
                index_type = 'flat'

            self._build_index(embeddings, index_type=index_type, nlist=nlist)
        else:
            if len(self.chunks) > nlist * 10 and not isinstance(self.index, faiss.IndexFlat):
                self.vprint(f"[VDB.add_documents] Too many chunks ({len(self.chunks)}). Convert to IVFFlat index to save memory.")
                self.convert_to_ivfflat(nlist=nlist)
            self.index.add(embeddings)

    def search(self, query, top_k=5):
        """
        Search for documents relevant to the query.

        Args:
            query (str): The search query string.
            top_k (int): The number of top relevant documents to retrieve.

        Returns:
            list of dict: A list of dictionaries representing the top_k search results,
                          each containing 'score' and 'chunk'.
        """
        self.vprint(f"[VDB.search] Searching for query: '{query}' (top_k={top_k})...")

        if self.index is None:
            raise ValueError("Index is not loaded or built.")

        # E-02: Add e5 prefixes correctly (query: for queries)
        query_embedding = self._embed_texts([query], prefix="query: ", batch_size=1, normalize_embeddings=True)

        distances, indices = self.index.search(query_embedding, top_k)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx != -1 and idx < len(self.chunks):
                results.append({
                    "score": float(dist),
                    "chunk": self.chunks[idx]
                })
        return results

    def migrate_to_ivf(self, nlist=100):
        """
        E-04: Migrate index from Flat to IVFFlat if not already IVF.

        Args:
            nlist (int): The number of cells/clusters for IVFFlat index.
        """
        self.vprint("[VDB.migrate_to_ivf] Migrating index from Flat to IVFFlat...")

        if not isinstance(self.index, faiss.IndexFlat):
            self.vprint("Index is already IVF or not initialized.")
            return

        n_total = self.index.ntotal
        all_vectors = self.index.reconstruct_n(0, n_total)

        quantizer = faiss.IndexFlatIP(self.dim)
        new_index = faiss.IndexIVFFlat(quantizer, self.dim, nlist, faiss.METRIC_INNER_PRODUCT)

        new_index.train(all_vectors)
        new_index.add(all_vectors)

        self.index = new_index

        self.vprint(f"Successfully migrated {n_total} vectors to IVFFlat.")

    def vprint(self, msg):
        if self.verbose:
            print(msg)

In [ ]:
def read_chunks(chunk_path, **kwargs):
    """
    Read chunks from a JSON file and create a vector database.

    Args:
        chunk_path (str): The file path to the JSON file containing chunks.

    Returns:
        vdb: The vector database containing the chunks.
    """
    db = vdb(**kwargs)

    with open(chunk_path, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    db.add_documents(chunks)

    return db

---

# RAG Generation Workflow

(Simryn)

In [ ]:
# R-01: retrieve()
# Assumes we already have:
# index (FAISS)
# embeddings_model
# chunks (list of text)

def retrieve(query, db, k=5):

    return db.search(query, top_k=k)

In [ ]:
# R-02: format_context()

def format_context(chunks):
    context = ""
    for i, chunk in enumerate(chunks):
        context += f"[Chunk {i+1}]\n{chunk}\n\n"
    return context

In [ ]:
# R-03: Mistral-7B-Instruct (4-bit)

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
# R-04: Study Guide Prompt

def study_guide_prompt(context, query):
    return f"""
You are an AI tutor.

Using the context below, create a structured study guide.

Context:
{context}

Question:
{query}

Output format:
- Key Concepts
- Definitions
- Important Points
- Summary
"""

Example 1:
Context: Neural networks use layers of neurons...
Question: Explain neural networks

Output:
Key Concepts:
- Layers
- Activation functions
...

Example 2:
Context: Overfitting occurs when...
Question: What is overfitting?

Output:
Key Concepts:
- Overfitting
...

In [ ]:
# R-05: Flashcards Prompt

def flashcard_prompt(context, query):
    return f"""
Generate flashcards from the context.

Return JSON array:
[
  {{"question": "...", "answer": "..."}}
]

Context:
{context}

Topic:
{query}
"""

[
  {"question": "What is overfitting?", "answer": "When a model memorizes training data"}
]

In [ ]:
# R-06: Practice Exam Prompt

def exam_prompt(context, query):
    return f"""
Create a practice exam.

Include:
- 3 multiple choice questions
- 2 short answer questions

Context:
{context}

Topic:
{query}
"""

In [ ]:
# R-07: ELI5 Prompt

def eli5_prompt(context, query):
    return f"""
Explain this like I'm 5 years old.

Context:
{context}

Question:
{query}
"""

In [ ]:
# create db class

# chunk_path = "{path}/chunks.json"
# db = read_chunks(chunk_path)

In [ ]:
# R-08: generate() (FULL PIPELINE)

def generate(query, mode, db, model, tokenizer):
    # Step 1: Retrieve
    retrieved_chunks = retrieve(query, db)

    # Step 2: Format
    context = format_context(retrieved_chunks)

    # Step 3: Choose prompt
    if mode == "study_guide":
        prompt = study_guide_prompt(context, query)
    elif mode == "flashcards":
        prompt = flashcard_prompt(context, query)
    elif mode == "exam":
        prompt = exam_prompt(context, query)
    elif mode == "eli5":
        prompt = eli5_prompt(context, query)
    else:
        raise ValueError("Invalid mode")

    # Step 4: Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# R-09: parse_flashcards()

import json
import re

def parse_flashcards(output):
    # Remove markdown fences
    cleaned = re.sub(r"```json|```", "", output).strip()

    return json.loads(cleaned)

In [ ]:
# R-10: export_anki_csv()

def export_anki_csv(flashcards, filename="anki_cards.csv"):
    with open(filename, "w") as f:
        for card in flashcards:
            f.write(f"{card['question']}\t{card['answer']}\n")

In [ ]:
# R-11: Testing

In [ ]:
# R-12: Parameter Tuning